## SRP421152

**paper:** [PMID: 38365592](https://pmc.ncbi.nlm.nih.gov/articles/PMC10874052/) - Distinct regulatory networks control toxin gene expression in elapid and viperid snakes, 2024

**date, curator:** 2026-03-06, Sara Carsanaro

**notes**
* they use same SRS ids for each animal, however based on methods these are not replicates
    * i added to the conditions section if the venom glands are milked or unmilked
    * i am not marking anything as replicates

### annotation summary

In [24]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Venom Gland,UBERON:0008976,snake venom gland,perfect match


In [25]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP421152"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 6/6 [00:05<00:00,  1.02it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes
Fetching RunIds and ReadHashes for each library...can take several minutes
⚠️ Failed to fetch ReadHash for SRX19290265: Command 'esearch -db sra -query SRX19290265 | efetch -format runinfo' returned non-zero exit status 1.


### library annnotations

In [11]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX19290268,SRP421152,Illumina HiSeq 2500,SRS16691280,,,,,,Venom Gland,Adult,,,,Unknown,,,8739,NEBNext Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,,05/03/2026,Small RNA library preparations were done with the NEBNext Small RNA Library Prep Set for Illumina (New England BioLabs) and AMPure XP Bead (Beckman Coulter) selection step for a 110-160 bp size range.,,C_viridis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348476,2BB1469808F8D1C0BF52671BD20F5A82
1,SRX19290267,SRP421152,Illumina HiSeq 2500,SRS16691280,,,,,,Venom Gland,Adult,,,,Unknown,,,8739,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,,05/03/2026,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,C_viridis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348477,D37423BFC0922396F661DDAF5712C62F
2,SRX19290266,SRP421152,Illumina HiSeq 2500,SRS16691279,,,,,,Venom Gland,Adult,,,,Unknown,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,,05/03/2026,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348478,C0BBFE9AE3EB55A4BA59A8AC4E651598
3,SRX19290265,SRP421152,Illumina HiSeq 2500,SRS16691279,,,,,,Venom Gland,Adult,,,,Unknown,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Milked Venom Gland,,,05/03/2026,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,,
4,SRX19290264,SRP421152,Illumina HiSeq 2500,SRS16691279,,,,,,Venom Gland,Adult,,,,Unknown,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,,05/03/2026,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,P_textilis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348480,94A76873654130586AD129C51BEF14A5
5,SRX19290263,SRP421152,Illumina HiSeq 2500,SRS16691279,,,,,,Venom Gland,Adult,,,,Unknown,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Milked Venom Gland,,,05/03/2026,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,P_textilis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348481,F5CB0FD34FA70BF36492EB563B30F7E3


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Venom Gland']


In [12]:

# all
library.loc[:,'anatId'] = 'UBERON:0008976'
library.loc[:,'anatName'] = 'snake venom gland'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX19290268,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,,,,Venom Gland,Adult,perfect match,not documented,,Unknown,,,8739,NEBNext Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,,05/03/2026,Small RNA library preparations were done with the NEBNext Small RNA Library Prep Set for Illumina (New England BioLabs) and AMPure XP Bead (Beckman Coulter) selection step for a 110-160 bp size range.,,C_viridis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348476,2BB1469808F8D1C0BF52671BD20F5A82
1,SRX19290267,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,,,,Venom Gland,Adult,perfect match,not documented,,Unknown,,,8739,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,,05/03/2026,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,C_viridis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348477,D37423BFC0922396F661DDAF5712C62F
2,SRX19290266,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,,,,Venom Gland,Adult,perfect match,not documented,,Unknown,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,,05/03/2026,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348478,C0BBFE9AE3EB55A4BA59A8AC4E651598
3,SRX19290265,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,,,,Venom Gland,Adult,perfect match,not documented,,Unknown,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Milked Venom Gland,,,05/03/2026,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,,
4,SRX19290264,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,,,,Venom Gland,Adult,perfect match,not documented,,Unknown,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,,05/03/2026,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,P_textilis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348480,94A76873654130586AD129C51BEF14A5
5,SRX19290263,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,,,,Venom Gland,Adult,perfect match,not documented,,Unknown,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Milked Venom Gland,,,05/03/20

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['Adult']


In [13]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX19290268,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,Unknown,,,8739,NEBNext Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,,05/03/2026,Small RNA library preparations were done with the NEBNext Small RNA Library Prep Set for Illumina (New England BioLabs) and AMPure XP Bead (Beckman Coulter) selection step for a 110-160 bp size range.,,C_viridis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348476,2BB1469808F8D1C0BF52671BD20F5A82
1,SRX19290267,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,Unknown,,,8739,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,,05/03/2026,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,C_viridis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348477,D37423BFC0922396F661DDAF5712C62F
2,SRX19290266,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,Unknown,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,,05/03/2026,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348478,C0BBFE9AE3EB55A4BA59A8AC4E651598
3,SRX19290265,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,Unknown,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Milked Venom Gland,,,05/03/2026,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,,
4,SRX19290264,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,Unknown,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,,05/03/2026,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,P_textilis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348480,94A76873654130586AD129C51BEF14A5
5,SRX19290263,SRP421152,Illumina HiSeq 2500,SRS16691279

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [14]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX19290268,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8739,NEBNext Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,,05/03/2026,Small RNA library preparations were done with the NEBNext Small RNA Library Prep Set for Illumina (New England BioLabs) and AMPure XP Bead (Beckman Coulter) selection step for a 110-160 bp size range.,,C_viridis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348476,2BB1469808F8D1C0BF52671BD20F5A82
1,SRX19290267,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8739,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,,05/03/2026,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,C_viridis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348477,D37423BFC0922396F661DDAF5712C62F
2,SRX19290266,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,,05/03/2026,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348478,C0BBFE9AE3EB55A4BA59A8AC4E651598
3,SRX19290265,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Milked Venom Gland,,,05/03/2026,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,,
4,SRX19290264,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,,05/03/2026,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,P_textilis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348480,94A76873654130586AD129C51BEF14A5
5,SRX19290263,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake ven

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [ ]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
#library.loc[:,'RNASelection'] = ''

# view
display_df(library)

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

    #libraryId        SRSId
2  SRX19290266  SRS16691279
3  SRX19290265  SRS16691279
4  SRX19290264  SRS16691279
5  SRX19290263  SRS16691279
0  SRX19290268  SRS16691280
1  SRX19290267  SRS16691280


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [15]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-06'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX19290268,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8739,NEBNext Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,SAC,2026-03-06,Small RNA library preparations were done with the NEBNext Small RNA Library Prep Set for Illumina (New England BioLabs) and AMPure XP Bead (Beckman Coulter) selection step for a 110-160 bp size range.,,C_viridis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348476,2BB1469808F8D1C0BF52671BD20F5A82
1,SRX19290267,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8739,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,,Milked Venom Gland,,SAC,2026-03-06,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,C_viridis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348477,D37423BFC0922396F661DDAF5712C62F
2,SRX19290266,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,SAC,2026-03-06,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,SRR23348478,C0BBFE9AE3EB55A4BA59A8AC4E651598
3,SRX19290265,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Milked Venom Gland,,SAC,2026-03-06,The small RNA library preps from the extracted and unextracted P. textilis venom glands were completed using Illuminas TruSeq Small RNA Sample Preparation protocol with 1 g of total RNA as input. The finished small RNA library was loaded onto a 6% PAGE gel (Invitrogen) and a band of ~170-320 bp was excised from the gel.,,P_textilis_VG,,,,,,TRANSCRIPTOMIC,size fractionation,,
4,SRX19290264,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,,Unmilked Venom Gland,,SAC,2026-03-06,"For mRNA libraries, 1 g of total RNA from each sample was subjected to Illuminas TruSeq RNA Sample Preparation v2 protocol.",,P_textilis_VG,,,,,,TRANSCRIPTOMIC,Oligo-dT,SRR23348480,94A76873654130586AD129C51BEF14A5
5,SRX19290263,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:00

#### comments

In [16]:
library.loc[:,'comment'] = 'PMID: 38365592'

#### save complete file with correct columns

In [17]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX19290268,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8739,NEBNext Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,PMID: 38365592,Milked Venom Gland,,SAC,2026-03-06
1,SRX19290267,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8739,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Crotalus viridis,SAMN33139131,,,,,,,PMID: 38365592,Milked Venom Gland,,SAC,2026-03-06
2,SRX19290266,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,PMID: 38365592,Unmilked Venom Gland,,SAC,2026-03-06
3,SRX19290265,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,PMID: 38365592,Milked Venom Gland,,SAC,2026-03-06
4,SRX19290264,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,PMID: 38365592,Unmilked Venom Gland,,SAC,2026-03-06
5,SRX19290263,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaja textilis,SAMN33139130,,,,,,,PMID: 38365592,Milked Venom Gland,,SAC,2026-03-06


### experiment annotations

In [18]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP421152,RNA (mRNA and microRNA) from extracted and unextracted snake venom glands,"We have used high-throughput RNA-sequencing to profile gene expression and microRNAs (miRNAs) between active (extracted) and resting (unextracted) venom glands in an elapid (Eastern Brown Snake, Pseudonaja textilis), and viperid snake (Crotalus viridis).",SRA,,,,"TruSeq RNA Sample Preparation Kit, TruSeq Small RNA Library Prep Kit",full_length,,PRJNA931953,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [19]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

6

In [20]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = "TruSeq RNA Sample Preparation Kit, TruSeq Small RNA Library Prep Kit, NEBNext Small RNA Library Prep Kit"
experiment.loc[:,'protocolType'] = "full_length"

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP421152,RNA (mRNA and microRNA) from extracted and unextracted snake venom glands,"We have used high-throughput RNA-sequencing to profile gene expression and microRNAs (miRNAs) between active (extracted) and resting (unextracted) venom glands in an elapid (Eastern Brown Snake, Pseudonaja textilis), and viperid snake (Crotalus viridis).",SRA,total,Bgee 1K,6,"TruSeq RNA Sample Preparation Kit, TruSeq Small RNA Library Prep Kit, NEBNext Small RNA Library Prep Kit",full_length,,PRJNA931953,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [21]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '38365592'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10874052'
experiment.loc[:,'DOI'] = '10.1186/s12864-024-10090-y'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP421152,RNA (mRNA and microRNA) from extracted and unextracted snake venom glands,"We have used high-throughput RNA-sequencing to profile gene expression and microRNAs (miRNAs) between active (extracted) and resting (unextracted) venom glands in an elapid (Eastern Brown Snake, Pseudonaja textilis), and viperid snake (Crotalus viridis).",SRA,total,Bgee 1K,6,"TruSeq RNA Sample Preparation Kit, TruSeq Small RNA Library Prep Kit, NEBNext Small RNA Library Prep Kit",full_length,,PRJNA931953,38365592,https://pmc.ncbi.nlm.nih.gov/articles/PMC10874052,10.1186/s12864-024-10090-y,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [22]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [23]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [26]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [27]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
55587,SRX158353,SRP014151,Illumina HiSeq 2000,SRS348925,UBERON:0002107,liver,UBERON:0000104,life cycle,,liver,NA,perfect match,not documented,perfect match,F,,,8663,,,polyA,,,Notechis scutatus (Tiger Snake) liver transcri...,SAMN01086234,,,,,,,"no PMID, no additional info",,,SAC,2026-03-05
55588,SRX158115,SRP014151,Illumina HiSeq 2000,SRS348821,UBERON:0002107,liver,UBERON:0000104,life cycle,,liver,NA,perfect match,not documented,perfect match,M,,,8663,,,polyA,,,Notechis scutatus (Tiger Snake) liver transcri...,SAMN01086130,,,,,,,"no PMID, no additional info",,,SAC,2026-03-05
55589,SRX19290268,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8739,NEBNext Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Crotalus ...,SAMN33139131,,,,,,,PMID: 38365592,Milked Venom Gland,,SAC,2026-03-06
55590,SRX19290267,SRP421152,Illumina HiSeq 2500,SRS16691280,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8739,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Crotalus ...,SAMN33139131,,,,,,,PMID: 38365592,Milked Venom Gland,,SAC,2026-03-06
55591,SRX19290266,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaj...,SAMN33139130,,,,,,,PMID: 38365592,Unmilked Venom Gland,,SAC,2026-03-06
55592,SRX19290265,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq Small RNA Library Prep Kit,full_length,miRNA,,,Model organism or animal sample from Pseudonaj...,SAMN33139130,,,,,,,PMID: 38365592,Milked Venom Gland,,SAC,2026-03-06
55593,SRX19290264,SRP421152,Illumina HiSeq 2500,SRS16691279,UBERON:0008976,snake venom gland,UBERON:0000113,post-juvenile adult stage,,Venom Gland,Adult,perfect match,not documented,perfect match,NA,,,8673,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Model organism or animal sample from Pseudonaj...,SAMN33139130,,,,,,,PMID: 38365592,Unmilked Venom Gland,,SAC,2026-03-06


In [28]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1076,DRP005385,RNA-seq analyses of an amphibious sea snake La...,Expression of chemosensory genes were investig...,SRA,partial,Bgee 1K,4,TruSeq RNA Sample Preparation Kit,full_length,,PRJDB7257,31506057,https://pmc.ncbi.nlm.nih.gov/articles/PMC6742997/,10.1098/rspb.2019.1828,,removed one DNA seq library
1077,SRP014151,Notechis scutatus Transcriptome or Gene expres...,Notechis scutatus Transcriptome from a male an...,SRA,total,Bgee 1K,2,,,,PRJNA170152,,,,,no paper unfortunately
1078,SRP421152,RNA (mRNA and microRNA) from extracted and une...,We have used high-throughput RNA-sequencing to...,SRA,total,Bgee 1K,6,"TruSeq RNA Sample Preparation Kit, TruSeq Smal...",full_length,,PRJNA931953,38365592,https://pmc.ncbi.nlm.nih.gov/articles/PMC10874052,10.1186/s12864-024-10090-y,,


### add annotations to git

In [29]:
! git pull

Already up to date.


In [30]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [31]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [32]:
! git add $git_experiment_path $git_library_path

In [33]:
! git commit -m $commit_message_exp

[develop e1cf2f4] adding annotated bulk experiment SRP421152
 2 files changed, 7 insertions(+)


In [34]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.99 KiB | 1.99 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   97ca282..e1cf2f4  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push